## simple_triton example

This notebook illustrates how to use the simple_triton package to perform inference on tiles from a whole-slide image using a Triton inference server.

Notes for running:
- Run this notebook in a container with `--network=host` so that it can reach the Triton container
- Mount the EfficientNetV2S.tensorflow savedmodel directory to the triton container
- Load the model (below)

In [ ]:
# install large_image with tile sources
!apt update
!apt install -y python3-openslide openslide-tools
!pip install /tf/notebooks/histomics_stream 'large_image[tiff,openslide]' \
  scikit_image --find-links https://girder.github.io/large_image_wheels

# install simple_triton
!pip install /tf/notebooks/simple_triton

## Create a histomics stream study

In [ ]:
import histomics_stream as hs
import os

# slide parameters
batch = 64
magnification = 20
tile = 224
overlap = 0
chunk = 1792
mask_threshold=0.5
wsi_path = "/tf/notebooks/TCGA-AN-A0G0-01Z-00-DX1.BE0BB5DF-DEDA-48D8-B5D8-2735C767F28F.svs"
mask_path = "/tf/notebooks/TCGA-AN-A0G0-01Z-00-DX1.BE0BB5DF-DEDA-48D8-B5D8-2735C767F28F.mask.png"

# create a histomics-stream study
study = dict(
    version="version-1",
    tile_height=tile,
    tile_width=tile,
    overlap_height=overlap,
    overlap_width=overlap,
    slides=dict(
        first=dict(
            filename=wsi_path,
            slide_name=os.path.splitext(os.path.split(wsi_path)[1])[0],
            chunk_height=chunk,
            chunk_width=chunk,
        ),
        second=dict(
            filename=wsi_path,
            slide_name=os.path.splitext(os.path.split(wsi_path)[1])[0],
            chunk_height=chunk,
            chunk_width=chunk,
        )
    )
)

# set resolution
resolution = hs.configure.FindResolutionForSlide(
    study, target_magnification=20, magnification_source="exact"
)

# generate 
masking = hs.configure.TilesByGridAndMask(
    study, mask_filename=mask_path, mask_threshold=mask_threshold
)

#apply resolution to slide
for slide in study["slides"].values():
    resolution(slide)
    masking(slide)

## Set parameters and load model

Parameters for this example include parameters for reading from the whole-slide image (magnification, tile size, tile overlap, mask file), the inference server (address), the model (model name, input/output dimensions and type), and the inference client.

We load the model and verify that the model state is "READY".

In [ ]:
import json
from google.protobuf.json_format import MessageToDict
import numpy as np
import tritonclient.grpc as grpcclient

# triton parameters
url = "localhost:8001"  # url for grpc access to tirton server
input_dtype = np.float32  # set input data type
model_name = "EfficientNetV2S.tensorflow"  # set model name
dimension_input = [tile, tile, 3]
dimension_output = 1280

# create triton client
client = grpcclient.InferenceServerClient(url=url, verbose=True)

# load tensorflow model
updated = {'platform': 'tensorflow_savedmodel',
           'input': [{'name': 'input_2', 'dataType': 'TYPE_FP32', 'dims': ['224', '224', '3']}],
           'output': [{'name': 'avg_pool', 'dataType': 'TYPE_FP32', 'dims': ['1280']}],
           'maxBatchSize': 256}
client.load_model("EfficientNetV2S.tensorflow", config=json.dumps(updated))

# check readiness
client.get_model_repository_index()

# deleting the client in main prevents conflicts with child process clients
del client

## Run the inference

In [ ]:
import multiprocessing
import multiprocessing.queues
import numpy as np
from simple_triton.inference import InferenceRunner
from simple_triton.sharded_tiles import ShardedTiles
from simple_triton.submitter import analyze, TimedQueue
import time

# inference parameters
limit = 10  # limit on number of pending requests per worker
workers = 32  # total number of Submitter workers
verbose = True  # set verbose as False

# start timer
start = time.time()

# create input, output queues
qout = TimedQueue()

# Start consumers
print(f"Creating {workers} workers")
shards = []
for w in range(workers):
    shard = ShardedTiles(study, batch, w, workers)
    shards.append(InferenceRunner(url, model_name, shard, qout, limit, verbose=False))
for s in shards:
    s.start()

# collecct results
print("Collecting results")
results = []
N = workers
while N:
    output, t_put, t_get = qout.get()
    if output is None:
        N-=1
        print(N)
    else:
        output["times"]["qout_put"] = t_put
        output["times"]["qout_get"] = t_get
        results.append(output)

# display elapsed time
print(f"Total elapsed time: {time.time()-start}")
analyze(results)

In [ ]:
def merge(batches):
    features = np.concatenate([b["result"][0] for b in batches])
    metadata = {k: np.concatenate([b["metadata"][k] for b in batches]) 
                for k in batches[0]["metadata"].keys()}
    return features, metadata

feature, metadata = merge(results)

Top level interface
- histomics stream study
- workers 
- server address
- model info & optimizations

how to check results
- track batches somehow - ensure they are completed (celery?)